# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library, referencing all data entities such as record sets and fields by their `@id` as per the Croissant standard.

### Dataset Source
Schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Get the metadata object, which supports attribute access
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")


## 2. Data Overview
Review the available record sets and their fields using their `@id`.

**Note:** All references use the entity `@id` as required by the MLCommons Croissant specification.

In [ ]:
# List all record sets in the dataset and their fields (all by @id)
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined in this dataset schema. Attempting to enumerate from distributions (files)...")
    # Sometimes, older schemas or non-fully-annotated Croissant files may not have recordSets explicitly
    available_record_sets = [r for r in dataset._metadata_json.get('recordSet', [])]
    print(f"recordSet entries from JSON: {[x.get('@id', '[no @id]') for x in available_record_sets]}")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        if hasattr(rs, 'fields'):
            for f in rs.fields:
                print(f"  |-- Field @id: {f.id}, name: {f.name}, dataType: {getattr(f, 'dataType', None)}")
        else:
            print("  (RecordSet has no fields defined)")
    print(f"Total record sets: {len(record_sets)}")

# Save record set @id list for later extraction
record_set_ids = [rs.id for rs in record_sets]


## 3. Data Extraction
Load data from each record set (referenced by `@id`) into a pandas DataFrame for further analysis.

Below, we demonstrate extracting all available record sets for exploration.

In [ ]:
dataframes = {}
if not record_set_ids:
    print("No record sets to extract records from. Please check the dataset schema for data accessibility.")
else:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, rows: {len(dataframes[record_set_id])}")
        except Exception as ex:
            print(f"Could not load records for RecordSet {record_set_id}: {ex}")
    if dataframes:
        preview_id = list(dataframes.keys())[0]
        print(f"Columns for RecordSet @id {preview_id}:")
        print(dataframes[preview_id].columns.tolist())
        display(dataframes[preview_id].head())
    else:
        print("No DataFrames were loaded from the available record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps using `@id` references for all entities. 

We demonstrate basic filtering, normalization, and grouping, referencing field `@id`s. Replace `<numeric_field_id>` and `<group_field_id>` as appropriate after verifying loaded columns.

In [ ]:
# Choose the primary RecordSet by its @id (edit if a different RecordSet is desired)
if not dataframes:
    print("No DataFrame available for EDA.")
else:
    main_record_set_id = list(dataframes.keys())[0]  # Select the first for demonstration
    df = dataframes[main_record_set_id]

    print(f"Exploring DataFrame for RecordSet @id: {main_record_set_id}")
    print("Columns:", df.columns.tolist())

    # Identify a numeric and a grouping field by their @ids or names.
    # For demonstration, select the first numeric-like column and a group column if any exist.
    sample_numeric_field = None
    sample_group_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            sample_numeric_field = col
            break
    
    # Try common group fields
    for col in df.columns:
        if any(substr in col.lower() for substr in ["group", "ward", "gender", "county", "region"]):
            sample_group_field = col
            break

    if sample_numeric_field is not None:
        # Filter and normalize by numeric field
        threshold = df[sample_numeric_field].quantile(0.75) if df[sample_numeric_field].dtype != 'O' else 0
        filtered_df = df[df[sample_numeric_field] > threshold].copy()
        print(f"Filtered records where '{sample_numeric_field}' > {threshold} (75th percentile): {len(filtered_df)} out of {len(df)}.")
        norm_field = f"{sample_numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[sample_numeric_field] - filtered_df[sample_numeric_field].mean()) / filtered_df[sample_numeric_field].std()
        display(filtered_df[[sample_numeric_field, norm_field]].head())
        if sample_group_field is not None:
            grouped_df = filtered_df.groupby(sample_group_field)[sample_numeric_field].mean().reset_index()
            print(f"Grouped by '{sample_group_field}', showing mean of '{sample_numeric_field}':")
            display(grouped_df.head())
        else:
            print("No group-by field found among columns.")
    else:
        print("No numeric field found in DataFrame for further analysis.")

## 5. Visualization
Visualize data distributions or relationships using field `@id`s.

Below, we demonstrate a histogram for the first numeric field and (if found) a group-wise barplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and sample_numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[sample_numeric_field], kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of '{sample_numeric_field}' in RecordSet {main_record_set_id}")
    plt.xlabel(sample_numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    if sample_group_field is not None:
        plt.figure(figsize=(8,5))
        sns.barplot(
            data=grouped_df,
            x=sample_group_field, y=sample_numeric_field,
            palette='viridis')
        plt.title(f"Mean {sample_numeric_field} by {sample_group_field}")
        plt.xlabel(sample_group_field)
        plt.ylabel(f"Mean of {sample_numeric_field}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric data to visualize.')

## 6. Conclusion
This notebook has shown how to load metadata, examine available record sets and fields using their `@id`, perform exploratory data analysis, and visualize results for a Croissant dataset using `mlcroissant`.

- All entities, including record sets and fields, are referenced by their unique `@id` as required by the MLCommons Croissant specification.
- The dataset encompasses ordered logistic regression outputs on climate adaptation and knowledge adoption predictors in Northern Kenya.
- You can modify filtering/grouping/visualization fields as appropriate for your analytic goals.

__Continue exploring the dataset with custom analyses as needed!__